In [11]:
import sys
import pandas as pd
 

In [12]:
PUNCT_STRIP = ".,;:!?¡¿\"'()[]{}«»…—-"
 

def detectar_convencion(df):
    has_hash = df["token"].str.startswith("##").any()
    has_underscore = df["token"].str.startswith("▁").any()
    if has_hash and not has_underscore:
        return "wordpiece"
    if has_underscore and not has_hash:
        return "sentencepiece"
    raise ValueError(
        "No se pudo detectar automáticamente la convención (se encontraron ambos "
        "marcadores, o ninguno). Especifícala manualmente con --convention."
    )
 
 
def reconstruir(df, convention="auto"):
    df = df.sort_values(["id_tweet", "id_vector"]).reset_index(drop=True)
 
    if convention == "auto":
        convention = detectar_convencion(df)
        print(f"Convención detectada automáticamente: {convention}", file=sys.stderr)
 
    if convention == "wordpiece":
        is_start = ~df["token"].str.startswith("##")
        strip_marker = lambda t: t[2:] if t.startswith("##") else t
    elif convention == "sentencepiece":
        is_start = df["token"].str.startswith("▁")
        strip_marker = lambda t: t[1:] if t.startswith("▁") else t
    elif convention == "bertin":
        is_start = df["token"].str.startswith("Ġ")
        strip_marker = lambda t: t[1:] if t.startswith("Ġ") else t
    else:
        raise ValueError(f"Convención desconocida: {convention}")
 
    df["_is_start"] = is_start
    df["_clean_token"] = df["token"].map(strip_marker)
    df["_word_id"] = df.groupby("id_tweet")["_is_start"].cumsum()
 
    grouped = (
        df.groupby(["id_tweet", "_word_id"])
        .agg(
            word=("_clean_token", lambda s: "".join(s)),
            region=("region", "first"),
            id_vector_start=("id_vector", "min"),
            id_vector_end=("id_vector", "max"),
            n_subtokens=("id_vector", "count"),
        )
        .reset_index()
        .drop(columns=["_word_id"])
    )
 
    grouped["word_normalized"] = (
        grouped["word"].str.lower().str.strip(PUNCT_STRIP)
    )
    # descarta palabras que quedaron vacías tras quitar puntuación (ej. emojis sueltos)
    grouped = grouped[grouped["word_normalized"].str.len() > 0].reset_index(drop=True)
 
    return grouped
 

def reportar_frecuencias(grouped, umbrales=(5, 10)):
    freq = grouped["word_normalized"].value_counts()
    print(f"\nPalabras completas reconstruidas (ocurrencias totales): {len(grouped)}", file=sys.stderr)
    print(f"Palabras únicas (tipos): {len(freq)}", file=sys.stderr)
    for th in umbrales:
        n = (freq >= th).sum()
        print(f"Tipos con frecuencia >= {th}: {n}", file=sys.stderr)
    return freq

In [13]:
def cargar_datos(model_name):
    data_og = pd.read_csv(f"output/tokens_{model_name}.csv")    
    data_og['token'] = data_og['token'].astype(str)
    return data_og


In [14]:

for model_name in ['mex_state', 'mex_large', 'beto', 'robertuito']:
    print("-" * 60)
    print(model_name)
    data_og = cargar_datos(model_name)
    grouped = reconstruir(data_og)
    reportar_frecuencias(grouped)
    grouped.to_csv(f"output/words_{model_name}.csv", index=False)

------------------------------------------------------------
mex_state


Convención detectada automáticamente: wordpiece

Palabras completas reconstruidas (ocurrencias totales): 1689613
Palabras únicas (tipos): 90459
Tipos con frecuencia >= 5: 18652
Tipos con frecuencia >= 10: 10613


------------------------------------------------------------
mex_large


Convención detectada automáticamente: wordpiece

Palabras completas reconstruidas (ocurrencias totales): 1689613
Palabras únicas (tipos): 90459
Tipos con frecuencia >= 5: 18652
Tipos con frecuencia >= 10: 10613


------------------------------------------------------------
beto


Convención detectada automáticamente: wordpiece

Palabras completas reconstruidas (ocurrencias totales): 1519455
Palabras únicas (tipos): 80541
Tipos con frecuencia >= 5: 17659
Tipos con frecuencia >= 10: 10050


------------------------------------------------------------
robertuito


Convención detectada automáticamente: sentencepiece

Palabras completas reconstruidas (ocurrencias totales): 1437100
Palabras únicas (tipos): 90408
Tipos con frecuencia >= 5: 17839
Tipos con frecuencia >= 10: 10040


In [15]:
model_name = "bertin"
print("-" * 60)
print(model_name)
data_og = cargar_datos(model_name)
grouped = reconstruir(data_og, convention='bertin')
reportar_frecuencias(grouped)
grouped.to_csv(f"output/words_{model_name}.csv", index=False)

------------------------------------------------------------
bertin



Palabras completas reconstruidas (ocurrencias totales): 1464321
Palabras únicas (tipos): 107375
Tipos con frecuencia >= 5: 18591
Tipos con frecuencia >= 10: 10415
